## Objetivo del proyecto

El objetivo de este proyecto es predecir la gravedad de los accidentes de tráfico en Madrid utilizando técnicas de Machine Learning, a partir de variables como la localización, las condiciones meteorológicas, el momento del accidente y las características de los implicados.

Este análisis busca identificar los factores que influyen en la gravedad de los accidentes y facilitar la toma de decisiones basada en datos.

## Contexto y valor de negocio

Este proyecto tiene como finalidad mejorar la gestión de recursos en el ámbito de la seguridad vial y la atención sanitaria.

La predicción de la gravedad de los accidentes permite:

- Optimizar la distribución de recursos médicos (ambulancias, personal sanitario)
- Mejorar los tiempos de respuesta ante emergencias
- Identificar zonas y condiciones de mayor riesgo
- Apoyar la toma de decisiones en políticas de prevención

De este modo, el análisis contribuye a reducir el impacto de los accidentes y mejorar la seguridad en la ciudad.

## Hipótesis iniciales

Se plantean las siguientes hipótesis sobre los factores que pueden influir en la gravedad de los accidentes:

- Las condiciones meteorológicas adversas aumentan la gravedad de los accidentes
- Los accidentes que involucran peatones presentan mayor gravedad
- La hora punta influye en la gravedad debido a la mayor densidad de tráfico
- El consumo de alcohol incrementa la probabilidad de accidentes graves
- Existen diferencias en la gravedad según el tipo de accidente

Estas hipótesis serán contrastadas mediante análisis exploratorio y modelado.

## Arquitectura del proyecto

El proyecto sigue un flujo estructurado de procesamiento y análisis de datos orientado a predecir la gravedad de los accidentes de tráfico.

1. Ingesta de datos
Datos de accidentes del Ayuntamiento de Madrid
Datos meteorológicos de OpenWeather
2. Procesamiento y limpieza
Tratamiento de valores nulos
Conversión de formatos
Normalización de datos (Pandas)
3. Transformación (ETL)
Generación de variables: festivo, hora punta, estación, etc.
Integración de datos de tráfico y clima
Creación de tablas analíticas
4. Visualización y análisis

Se utiliza Power BI para analizar los datos y detectar los factores que influyen en la gravedad de los accidentes.
Esta fase permite validar hipótesis y orientar el modelado.

5. Modelado
Entrenamiento de modelos de Machine Learning
Optimización de hiperparámetros
Manejo del desbalanceo
6. Evaluación
Comparación de modelos mediante F1-score
Selección del mejor modelo

# 01 ETL y preparación de datos


En este notebook se realiza el proceso completo de ETL y preparación de datos para el análisis de accidentes de tráfico en Madrid.

Se parte de datos brutos de accidentes, que se cargan y unifican en un único dataset. Posteriormente, se lleva a cabo una limpieza básica, eliminando duplicados, gestionando valores nulos y corrigiendo tipos de datos, especialmente en variables temporales como fecha y hora.

A continuación, se integran datos meteorológicos mediante una API externa, realizando un merge por fecha y hora para enriquecer el dataset con variables como temperatura, precipitación y viento.

Se construye la variable objetivo “gravedad” a partir del nivel de lesividad, transformándola en un problema de clasificación binaria (grave / no grave).

Posteriormente, se realiza codificación de variables categóricas (one-hot encoding) y se generan nuevas variables relevantes, como alcohol positivo o edad media.

El dataset se transforma de nivel persona/vehículo a nivel accidente mediante una agregación por número de expediente, consolidando la información relevante.

Finalmente, se crean features temporales adicionales (festivos, estación, hora punta, día/noche, etc.) y se realiza una limpieza final para obtener un dataset listo para el modelado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, year, month, dayofweek

In [ ]:
spark = SparkSession.builder \
    .appName("ETL Accidentes") \
    .getOrCreate()

## 1. Carga de datos raw

In [ ]:
# Ajusta esta ruta a tu estructura real
RAW_GLOB = "/content/drive/MyDrive/Proyecto final Keepcoding/datasets/raw data/*.xlsx"

archivos = sorted(glob.glob(RAW_GLOB))
print(f"Archivos encontrados: {len(archivos)}")

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos con el patrón: {RAW_GLOB}")

dataframes = []
for archivo in archivos:
    print("Cargando:", Path(archivo).name)
    df_tmp = pd.read_excel(archivo)
    dataframes.append(df_tmp)

df_raw = pd.concat(dataframes, ignore_index=True)
print("Shape raw:", df_raw.shape)
df_raw.head()

Archivos encontrados: 7
Cargando: 300228-0-accidentes-trafico-detalle-xlsx.xlsx
Cargando: 300228-31-accidentes-trafico-detalle-xlsx.xlsx
Cargando: 300228-35-accidentes-trafico-detalle.xlsx
Cargando: 300228-4-accidentes-trafico-detalle-xlsx.xlsx
Cargando: 300228-6-accidentes-trafico-detalle-xlsx.xlsx
Cargando: 300228-7-accidentes-trafico-detalle-xlsx.xlsx
Cargando: 300228-9-accidentes-trafico-detalle-xlsx.xlsx
Shape raw: (274804, 19)


,num_expediente,fecha,hora,localizacion,numero,cod_distrito,distrito,tipo_accidente,estado_meteorológico,tipo_vehiculo,tipo_persona,rango_edad,sexo,cod_lesividad,lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga
0,2023S040280,2024-01-04,14:09:00,AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11,3,16.0,HORTALEZA,Colisión fronto-lateral,Lluvia débil,Motocicleta > 125cc,Conductor,De 55 a 59 años,Hombre,2.0,Ingreso inferior o igual a 24 horas,444913.056,4481427.179,N,NaN
1,2023S040280,2024-01-04,14:09:00,AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11,3,16.0,HORTALEZA,Colisión fronto-lateral,Lluvia débil,Turismo,Conductor,De 55 a 59 años,Mujer,14.0,Sin asistencia sanitaria,444913.056,4481427.179,N,NaN
2,2023S040309,2024-02-15,14:05:00,CALL. TESORO / CALL. MINAS,18,1.0,CENTRO,Colisión fronto-lateral,Lluvia débil,Bicicleta,Conductor,De 25 a 29 años,Hombre,7.0,Asistencia sanitaria sólo en el lugar del acci...,440122.814,4475169.548,N,NaN
3,2023S040309,2024-02-15,14:05:00,CALL. TESORO / CALL. MINAS,18,1.0,CENTRO,Colisión fronto-lateral,Lluvia débil,Motocicleta hasta 125cc,Conductor,De 35 a 39 años,Hombre,14.0,Sin asistencia sanitaria,440122.814,4475169.548,N,NaN
4,2023S040310,2024-02-18,10:40:00,GTA. RUIZ JIMENEZ / CALL. SAN BERNARDO,3,7.0,CHAMBERÍ,Colisión lateral,Despejado,Turismo,Conductor,De 25 a 29 años,Hombre,NaN,NaN,440136.981,4475721.316,N,NaN


In [ ]:
df_raw.dtypes

,0
num_expediente,object
fecha,datetime64[ns]
hora,object
localizacion,object
numero,object
cod_distrito,float64
distrito,object
tipo_accidente,object
estado_meteorológico,object
tipo_vehiculo,object


In [ ]:
df_raw["hora"] = df_raw["hora"].astype(str)

In [ ]:
df_spark = spark.createDataFrame(df_raw)

## 2. Limpieza básica y revisión de nulos

In [ ]:
df_spark = df_spark.dropDuplicates()

non_empty_cols = [c for c in df_spark.columns if df_spark.filter(col(c).isNotNull()).count() > 0]
df_spark = df_spark.select(non_empty_cols)

df = df_spark.toPandas()

print("Shape tras limpieza básica:", df.shape)
display(df.info())
display((df.isnull().mean() * 100).sort_values(ascending=False))

Shape tras limpieza básica: (263181, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263181 entries, 0 to 263180
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   num_expediente        263181 non-null  object        
 1   fecha                 263181 non-null  datetime64[ns]
 2   hora                  263181 non-null  object        
 3   localizacion          263181 non-null  object        
 4   numero                263181 non-null  object        
 5   cod_distrito          263173 non-null  float64       
 6   distrito              263181 non-null  object        
 7   tipo_accidente        263181 non-null  object        
 8   estado_meteorológico  263181 non-null  object        
 9   tipo_vehiculo         263181 non-null  object        
 10  tipo_persona          263181 non-null  object        
 11  rango_edad            263181 non-null  object        
 12  sexo             

None

,0
positiva_droga,99.669809
cod_lesividad,44.053712
coordenada_y_utm,0.022798
coordenada_x_utm,0.022798
cod_distrito,0.003040
num_expediente,0.000000
fecha,0.000000
hora,0.000000
localizacion,0.000000
numero,0.000000


In [ ]:
# Eliminamos columna con casi todo nulo
if "positiva_droga" in df.columns:
    df = df.drop(columns=["positiva_droga"])

print("Columnas actuales:")
print(df.columns.tolist())

Columnas actuales:
['num_expediente', 'fecha', 'hora', 'localizacion', 'numero', 'cod_distrito', 'distrito', 'tipo_accidente', 'estado_meteorológico', 'tipo_vehiculo', 'tipo_persona', 'rango_edad', 'sexo', 'cod_lesividad', 'lesividad', 'coordenada_x_utm', 'coordenada_y_utm', 'positiva_alcohol']


## 3. Tipado correcto de fecha y hora

In [ ]:
df_spark = spark.createDataFrame(df)

df_spark = df_spark.withColumn("fecha", to_date(col("fecha")))
df_spark = df_spark.withColumn("año", year(col("fecha")))
df_spark = df_spark.withColumn("mes", month(col("fecha")))
df_spark = df_spark.withColumn("dia_semana", dayofweek(col("fecha")))

df = df_spark.toPandas()

In [ ]:
# Fecha
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

# Hora robusta: admite formatos como "03:00:00", "3", datetime, etc.
hora_str = df["hora"].astype(str).str.extract(r"(\d{1,2})")[0]
df["hora"] = pd.to_numeric(hora_str, errors="coerce").astype("Int64")

# Derivadas temporales
df["año"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df["dia_semana"] = df["fecha"].dt.dayofweek

display(df[["fecha", "hora", "año", "mes", "dia_semana"]].head())
display(df[["fecha", "hora"]].isnull().mean() * 100)

,fecha,hora,año,mes,dia_semana
0,2024-01-02,7,2024,1,1
1,2024-01-04,0,2024,1,3
2,2024-01-05,7,2024,1,4
3,2024-01-09,14,2024,1,1
4,2024-01-11,14,2024,1,3


,0
fecha,0.0
hora,0.0


## 4. Descarga de clima y merge por fecha + hora

In [ ]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 40.4168,
    "longitude": -3.7038,
    "start_date": str(df["fecha"].min().date()),
    "end_date": str(df["fecha"].max().date()),
    "hourly": ["temperature_2m", "precipitation", "windspeed_10m"],
    "timezone": "Europe/Madrid"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data = response.json()

clima = pd.DataFrame(data["hourly"])
clima["time"] = pd.to_datetime(clima["time"], errors="coerce")
clima["fecha"] = clima["time"].dt.normalize()
clima["hora"] = clima["time"].dt.hour.astype("Int64")

clima = clima[["fecha", "hora", "temperature_2m", "precipitation", "windspeed_10m"]]
print(clima.shape)
clima.head()

(53352, 5)


,fecha,hora,temperature_2m,precipitation,windspeed_10m
0,2020-01-01,0,3.5,0.0,4.3
1,2020-01-01,1,2.4,0.0,5.5
2,2020-01-01,2,1.3,0.0,7.2
3,2020-01-01,3,1.5,0.0,6.2
4,2020-01-01,4,0.7,0.0,5.4


In [ ]:
# Aseguramos mismo tipo de fecha
df["fecha"] = pd.to_datetime(df["fecha"]).dt.normalize()

df = df.merge(clima, on=["fecha", "hora"], how="left")

display(df[["temperature_2m", "precipitation", "windspeed_10m"]].isnull().mean() * 100)
df.head()

,0
temperature_2m,0.0
precipitation,0.0
windspeed_10m,0.0


,num_expediente,fecha,hora,localizacion,numero,cod_distrito,distrito,tipo_accidente,estado_meteorológico,tipo_vehiculo,...,lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,año,mes,dia_semana,temperature_2m,precipitation,windspeed_10m
0,2024S000056,2024-01-02,7,"AEROP. TERMINAL T-4, LLEGADAS, LINEA DE TAXIS",0,21.0,BARAJAS,Atropello a persona,Despejado,Turismo,...,Sin asistencia sanitaria,449848.973,4482595.779,N,2024,1,1,3.2,0.0,3.1
1,2024S000215,2024-01-04,0,CALL. TOLEDO / CALL. CALATRAVA,105,1.0,CENTRO,Colisión fronto-lateral,Nublado,VMU eléctrico,...,Asistencia sanitaria sólo en el lugar del acci...,439743.331,4473412.020,N,2024,1,3,9.1,0.0,5.6
2,2024S000304,2024-01-05,7,"PLAZA. SANTA ANA, 6",6,1.0,CENTRO,Colisión múltiple,Despejado,Turismo,...,NaN,440550.589,4474000.896,N,2024,1,4,5.1,0.2,15.9
3,2024S000802,2024-01-09,14,"CALL. SERRANO, 37",37,4.0,SALAMANCA,Colisión múltiple,Nublado,Turismo,...,Asistencia sanitaria sólo en el lugar del acci...,441690.368,4475512.486,N,2024,1,1,7.2,0.0,4.4
4,2024S000911,2024-01-11,14,AVDA. DEMOCRACIA / AVDA. ALBUFERA,3,13.0,PUENTE DE VALLECAS,Colisión fronto-lateral,Despejado,Turismo,...,NaN,446885.697,4470520.562,N,2024,1,3,6.4,0.0,9.6


## 5. Construcción del target `gravedad`

In [ ]:
# Mapeo binario: 0 = no grave, 1 = grave

map_gravedad = {
    "Sin asistencia sanitaria": 0,
    "Asistencia sanitaria sólo en el lugar del accidente": 1,
    "Atención en urgencias sin posterior ingreso": 1,
    "Asistencia sanitaria ambulatoria con posterioridad": 1,
    "Asistencia sanitaria inmediata en centro de salud o mutua": 1,
    "Ingreso inferior o igual a 24 horas": 1,
    "Ingreso superior a 24 horas": 1,
    "Fallecido 24 horas": 1
}

df["gravedad"] = df["lesividad"].map(map_gravedad)

display(df["lesividad"].value_counts(dropna=False))
display(df["gravedad"].value_counts(dropna=False))

,count
lesividad,
NaN,115941
Sin asistencia sanitaria,83060
Asistencia sanitaria sólo en el lugar del accidente,33254
Ingreso inferior o igual a 24 horas,10382
Atención en urgencias sin posterior ingreso,7957
Asistencia sanitaria inmediata en centro de salud o mutua,6198
Asistencia sanitaria ambulatoria con posterioridad,3115
Ingreso superior a 24 horas,3086
Fallecido 24 horas,177


,count
gravedad,
NaN,115952
0.0,83060
1.0,64169


## 6. One-hot antes de agrupar para no perder información

Esto tiene sentido porque el dataset original está a nivel persona/vehículo.
Luego agregaremos por accidente usando `max` en esas dummies.

In [ ]:
cols_ohe = [c for c in ["tipo_vehiculo", "tipo_accidente", "sexo", "tipo_persona"] if c in df.columns]
df = pd.get_dummies(df, columns=cols_ohe, drop_first=False, dtype=int)

print("Shape tras dummies:", df.shape)

Shape tras dummies: (263181, 84)


## 7. Variables derivadas útiles

In [ ]:
# Alcohol positivo
df_spark = spark.createDataFrame(df)

df_spark = df_spark.withColumn(
    "alcohol_positivo",
    when(col("positiva_alcohol") == "S", 1).otherwise(0)
)

df = df_spark.toPandas()

# Edad media desde rango_edad
def extraer_edad_media(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    if texto.strip().lower() == "desconocido":
        return np.nan
    nums = list(map(int, __import__("re").findall(r"\d+", texto)))
    if len(nums) >= 2:
        return sum(nums[:2]) / 2
    if len(nums) == 1:
        return float(nums[0])
    return np.nan

if "rango_edad" in df.columns:
    df["edad_media"] = df["rango_edad"].apply(extraer_edad_media)
else:
    df["edad_media"] = np.nan

df[["alcohol_positivo", "edad_media"]].head()

,alcohol_positivo,edad_media
0,0,62.0
1,0,27.0
2,0,42.0
3,0,57.0
4,0,27.0


## 8. Agrupación de vehículos en familias

In [ ]:
cols_vehiculo = [c for c in df.columns if c.startswith("tipo_vehiculo_")]

def cols_contienen(patrones):
    return [c for c in cols_vehiculo if any(p in c for p in patrones)]

df["vehiculo_coche"] = df[cols_contienen(["Turismo", "Todo terreno"])].sum(axis=1)
df["vehiculo_moto"] = df[cols_contienen(["Motocicleta", "Ciclomotor", "Moto de tres ruedas", "Cuadriciclo", "Ciclo de motor"])].sum(axis=1)
df["vehiculo_bici_patinete"] = df[cols_contienen(["Bicicleta", "EPAC", "Ciclo", "Patinete"])].sum(axis=1)
df["vehiculo_bus"] = df[cols_contienen(["Autobús", "Autobus", "Microbús", "Bus"])].sum(axis=1)
df["vehiculo_furgoneta"] = df[cols_contienen(["Furgoneta"])].sum(axis=1)
df["vehiculo_pesado"] = df[cols_contienen(["Camión", "Tractocamión", "Remolque", "Semiremolque", "Maquinaria", "Caravana"])].sum(axis=1)
df["vehiculo_especial"] = df[cols_contienen(["Ambulancia", "Bomberos"])].sum(axis=1)
df["vehiculo_otros"] = df[cols_contienen(["Tren", "Metro", "Sin especificar"])].sum(axis=1)

# Eliminamos dummies detalladas de tipo_vehiculo
df = df.drop(columns=cols_vehiculo, errors="ignore")

## 9. Agregación a nivel accidente (`num_expediente`)

In [ ]:
# Columnas auxiliares de dummies no vehículo
cols_tipo_acc = [c for c in df.columns if c.startswith("tipo_accidente_")]
cols_sexo = [c for c in df.columns if c.startswith("sexo_")]
cols_tipo_persona = [c for c in df.columns if c.startswith("tipo_persona_")]

agg_dict = {
    "gravedad": "max",
    "fecha": "first",
    "hora": "first",
    "distrito": "first",
    "estado_meteorológico": "first",
    "temperature_2m": "mean",
    "precipitation": "mean",
    "windspeed_10m": "mean",
    "coordenada_x_utm": "first",
    "coordenada_y_utm": "first",
    "alcohol_positivo": "max",
    "edad_media": "mean",
    "dia_semana": "first",
    "mes": "first",
    "vehiculo_coche": "sum",
    "vehiculo_moto": "sum",
    "vehiculo_bici_patinete": "sum",
    "vehiculo_bus": "sum",
    "vehiculo_furgoneta": "sum",
    "vehiculo_pesado": "sum",
    "vehiculo_especial": "sum",
    "vehiculo_otros": "sum",
}

for c in cols_tipo_acc + cols_sexo + cols_tipo_persona:
    agg_dict[c] = "max"

# Asegura que solo usamos columnas presentes
agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}

df_acc = df.groupby("num_expediente", as_index=False).agg(agg_dict)

print("Shape agregado:", df_acc.shape)
display(df_acc["gravedad"].value_counts(dropna=False))
df_acc.head()

Shape agregado: (117243, 45)


,count
gravedad,
1.0,52494
NaN,49614
0.0,15135


,num_expediente,gravedad,fecha,hora,distrito,estado_meteorológico,temperature_2m,precipitation,windspeed_10m,coordenada_x_utm,...,tipo_accidente_Solo salida de la vía,tipo_accidente_Vuelco,sexo_Desconocido,sexo_Hombre,sexo_Mujer,tipo_persona_Conductor,tipo_persona_NaN,tipo_persona_Pasajero,tipo_persona_Peatón,tipo_persona_Peatón (atropello sc)
0,2019S040008,NaN,2020-09-07,23,CIUDAD LINEAL,Despejado,20.7,0.0,18.6,444578.153,...,0,0,0,1,1,1,0,0,0,0
1,2020S000001,NaN,2020-01-01,1,SAN BLAS-CANILLEJAS,NaN,2.4,0.0,5.5,447894.521,...,0,0,0,1,0,1,0,0,0,0
2,2020S000002,NaN,2020-01-01,1,HORTALEZA,Despejado,2.4,0.0,5.5,445094.901,...,0,0,0,1,0,1,0,1,0,0
3,2020S000003,NaN,2020-01-01,2,CHAMBERÍ,Despejado,1.3,0.0,7.2,440277.207,...,0,0,0,1,1,1,0,1,0,0
4,2020S000004,NaN,2020-01-01,1,CIUDAD LINEAL,Despejado,2.4,0.0,5.5,445423.835,...,0,0,1,1,0,1,0,0,0,0


## 10. Festivos y nuevas features temporales

In [ ]:
import pandas as pd
import holidays

# 1. Crear calendario de festivos (IMPORTANTE incluir años)
es_holidays = holidays.country_holidays(
    "ES",
    subdiv="MD",
    years=range(2019, 2027)
)

# 2. Convertir la fecha correctamente
df_acc["fecha_date"] = pd.to_datetime(df_acc["fecha"]).dt.date

# 3. Convertir festivos a set (más eficiente)
festivos_set = set(es_holidays.keys())

# 4. Crear variable festivo
df_acc["festivo"] = df_acc["fecha_date"].isin(festivos_set).astype(int)

In [ ]:
df_acc["festivo"].value_counts()

,count
festivo,
0,114611
1,2632


In [ ]:


def obtener_estacion(mes):
    if mes in [12, 1, 2]:
        return "invierno"
    elif mes in [3, 4, 5]:
        return "primavera"
    elif mes in [6, 7, 8]:
        return "verano"
    return "otoño"

df_acc["estacion"] = df_acc["mes"].apply(obtener_estacion)
df_acc["fin_semana"] = (pd.to_datetime(df_acc["fecha"]).dt.dayofweek >= 5).astype(int)
df_acc["hora_punta"] = df_acc["hora"].isin([7, 8, 9, 18, 19, 20]).astype(int)
df_acc["noche"] = (((df_acc["hora"] >= 22) | (df_acc["hora"] <= 6))).astype(int)

df_acc["franja_horaria"] = pd.cut(
    df_acc["hora"],
    bins=[-1, 6, 12, 18, 23],
    labels=["madrugada", "mañana", "tarde", "noche"]
)

## 11. Limpieza final

In [ ]:
# Tratamiento de nulos final
df_acc["estado_meteorológico"] = df_acc["estado_meteorológico"].fillna("Desconocido")

# Eliminamos filas sin target porque no hay base fiable para imputarlo
df_acc = df_acc.dropna(subset=["gravedad"]).copy()

# En variables con pocos nulos, quitamos observaciones residuales
df_acc = df_acc.dropna(subset=["coordenada_x_utm", "coordenada_y_utm", "edad_media", "distrito"])

# Ajustes de tipo
df_acc["gravedad"] = df_acc["gravedad"].astype(int)
df_acc["hora"] = df_acc["hora"].astype(int)
df_acc["dia_semana"] = df_acc["dia_semana"].astype(int)
df_acc["mes"] = df_acc["mes"].astype(int)

display((df_acc.isnull().mean() * 100).sort_values(ascending=False).head(20))
print(df_acc.shape)
df_acc.head()

,0
num_expediente,0.0
gravedad,0.0
fecha,0.0
hora,0.0
distrito,0.0
estado_meteorológico,0.0
temperature_2m,0.0
precipitation,0.0
windspeed_10m,0.0
coordenada_x_utm,0.0


(67542, 52)


,num_expediente,gravedad,fecha,hora,distrito,estado_meteorológico,temperature_2m,precipitation,windspeed_10m,coordenada_x_utm,...,tipo_persona_Pasajero,tipo_persona_Peatón,tipo_persona_Peatón (atropello sc),fecha_date,festivo,estacion,fin_semana,hora_punta,noche,franja_horaria
6,2020S000006,0,2020-01-01,5,CARABANCHEL,Despejado,0.2,0.0,6.6,436672.459,...,1,0,0,2020-01-01,1,invierno,0,0,1,madrugada
7,2020S000007,1,2020-01-01,6,SALAMANCA,Despejado,-0.6,0.0,6.1,443740.487,...,1,0,0,2020-01-01,1,invierno,0,0,1,madrugada
12,2020S000015,0,2020-01-01,2,CHAMARTÍN,Despejado,1.3,0.0,7.2,441461.761,...,1,0,0,2020-01-01,1,invierno,0,0,1,madrugada
13,2020S000017,0,2020-01-01,3,PUENTE DE VALLECAS,Despejado,1.5,0.0,6.2,442141.822,...,0,0,0,2020-01-01,1,invierno,0,0,1,madrugada
15,2020S000020,1,2020-01-01,9,CARABANCHEL,Despejado,-1.6,0.0,7.4,439691.436,...,1,0,0,2020-01-01,1,invierno,0,1,0,mañana


## 12. Guardado del dataset procesado

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/Proyecto final Keepcoding/datasets/processed data/accidentes_madrid_procesado.csv"

df_acc.to_csv(OUTPUT_PATH, index=False)

print("Notebook ETL corregido listo.")
print("Ruta sugerida de salida:", OUTPUT_PATH)

Notebook ETL corregido listo.
Ruta sugerida de salida: /content/drive/MyDrive/Proyecto final Keepcoding/datasets/processed data/accidentes_madrid_procesado.csv
